# Week 5 · 05 Optimizers: what to do with the gradient

`03` computes the gradient; this notebook is about how to use it. Plain gradient descent has one knob, the learning rate, and two problems: it zigzags in narrow valleys, and computing the gradient on all the data for every step is expensive. Minibatches (SGD), momentum, RMSProp and Adam fix these one at a time. Each update rule is two or three lines of NumPy; you write them all, then check them against `torch.optim` number for number. Reading: ISL §10.7.2.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **learning rate $\eta$**: the step size: how far each update moves the weights along the (negative) gradient.
- **minibatch**: a small random subset of the training data (e.g. 32 samples) used to estimate the gradient for one step.
- **step (iteration)**: one update of the weights.
- **epoch**: one pass over the whole training set: $\lceil n / b \rceil$ steps with batches of size $b$.
- **momentum**: a running average of past gradients, used as the step direction.
- **EWMA**: exponentially weighted moving average: $a_t = \beta a_{t-1} + (1 - \beta) x_t$, an average that forgets old values gradually.
- **schedule**: a rule that changes the learning rate during training.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "05")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** Each method fixes one problem of the one before it, and Adam combines the fixes.

In [ ]:
fig_opt_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and the numerical gradient from memory | `03` §6, a few days later |
| E | four optimizers on one valley | start from the end: what each one does, before how |
| 1 | gradient descent and the learning rate | the one knob, with a proof of when it converges |
| 2 | minibatches (SGD) | a gradient from 32 samples instead of all of them: why that's allowed |
| 3 | momentum | averaging the step directions to stop the zigzag |
| 4 | RMSProp | a separate step size for every weight |
| 5 | Adam | momentum + RMSProp + a correction for the start; the default optimizer |
| 6 | schedules | changing the learning rate during training |
| 7 | `torch.optim` | the same numbers in PyTorch, and your own Adam class |

## 0. Recap first · 10 min, closed book

**Quick check:** The gradient descent update, and what $\eta$ is?

<details><summary>Answer</summary>

$w \leftarrow w - \eta\, \nabla L(w)$; $\eta$ is the learning rate (step size). [Week 1 · 03 §7 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** After `loss.backward()` in PyTorch, where are the gradients?

<details><summary>Answer</summary>

In each parameter's `.grad`. They **accumulate** over calls, so they must be zeroed before every step. [04 §3 ↗](04_micrograd.ipynb)

</details>

**Quick check:** Which $\varepsilon$ for a centered numerical derivative in float64, and why not much smaller?

<details><summary>Answer</summary>

About $10^{-5}$. Much smaller and rounding error (about $10^{-16} / \varepsilon$) dominates. [03 §6 ↗](03_backprop.ipynb)

</details>

**Quick check:** `rng.permutation(5)` returns ...?

<details><summary>Answer</summary>

The numbers 0–4 in a random order, e.g. `[2 0 4 1 3]`: each exactly once.

</details>

**Quick check:** A weighted average $a_t = 0.9\, a_{t-1} + 0.1\, x_t$ with $a_0 = 0$ and every $x_t = 1$. What is $a_1$? $a_2$?

<details><summary>Answer</summary>

$a_1 = 0.1$, $a_2 = 0.19$: it starts far below 1 and creeps up. Adam corrects exactly this start (§5).

</details>

**Exercise 1 · Numerical gradient, from memory.** Return an array shaped like `theta` with $\frac{f(\theta + \varepsilon e) - f(\theta - \varepsilon e)}{2\varepsilon}$ at each position (`np.ndindex(theta.shape)` visits every position). Put each entry back afterwards: `theta` must end unchanged.

**Example**: what goes in, and what should come out:
```python
numerical_grad(lambda t: float((t ** 2).sum()), np.array([1.0, -2.0]))    # → [ 2. -4.]
```

In [ ]:
def numerical_grad(f, theta, eps=1e-5):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: four optimizers on one valley

The test problem: $f(w) = \frac{1}{2}(w_1^2 + 100\, w_2^2)$, a long narrow valley (steep across, shallow along), starting at $(-4, 1)$. Its minimum is $(0, 0)$. Each optimizer gets a sensible setting; we count steps until $f < 0.001$.

🖼 **Picture.** The first 60 steps of each. Gradient descent zigzags across and crawls along. Momentum swings, but the swings die out while it keeps moving along. RMSProp takes one big first step (§4 explains it), then walks straight down the valley. Adam curves in and circles the minimum before settling.

In [ ]:
fig_valley_paths()

| Optimizer | Update (§1–5) | Keeps between steps | Steps to $f < 0.001$ |
|---|---|---|---|
| gradient descent, $\eta = 0.019$ | $w \leftarrow w - \eta g$ | nothing | 235 |
| momentum, $\eta = 0.005$, $\beta = 0.9$ | $v \leftarrow \beta v + g$; $\ w \leftarrow w - \eta v$ | $v$ (a velocity) | 91 |
| RMSProp, $\eta = 0.1$, $\rho = 0.99$ | $s \leftarrow \rho s + (1 - \rho) g^2$; $\ w \leftarrow w - \eta\, g / \sqrt{s}$ | $s$ (recent gradient sizes) | 25 |
| Adam, $\eta = 0.3$ | momentum + RMSProp + bias correction | $m$, $s$, and the step count $t$ | 61 |

In PyTorch, the optimizer is one line, and swapping it changes nothing else in the training loop. The same race in PyTorch (`06` explains the loop):

In [ ]:
import torch

def race(make_optimizer, max_steps=1000):
    w = torch.tensor([-4.0, 1.0], dtype=torch.float64, requires_grad=True)
    opt = make_optimizer([w])                       # an optimizer is given the list of tensors it may change
    for t in range(max_steps):
        f = 0.5 * (w[0] ** 2 + 100 * w[1] ** 2)
        if f.item() < 1e-3:
            return t
        opt.zero_grad()                             # clear w.grad
        f.backward()                                # fill w.grad
        opt.step()                                  # change w using w.grad (and the optimizer's own memory)
    return max_steps

print(race(lambda p: torch.optim.SGD(p, lr=0.019)),
      race(lambda p: torch.optim.SGD(p, lr=0.005, momentum=0.9)),
      race(lambda p: torch.optim.RMSprop(p, lr=0.1)),
      race(lambda p: torch.optim.Adam(p, lr=0.3)))      # → 235 91 25 61

**Don't read this as a ranking.** This valley is lined up with the axes, which is the ideal case for RMSProp's per-weight step sizes; on real networks the differences are smaller, and Adam is the usual default because it's the least sensitive to its learning rate. **The rest of this notebook answers:** what does each update do, and why does it help?

## 1. Gradient descent and the learning rate

**The question:** how large may the step be? Take the simplest case, $f(w) = w^2$, gradient $2w$. One step is $w_{t+1} = w_t - \eta \cdot 2 w_t = (1 - 2\eta)\, w_t$, so $w_t = (1 - 2\eta)^t\, w_0$. Starting at $w_0 = 1$:

| $\eta$ | $1 - 2\eta$ | $w_0, w_1, w_2, w_3$ | What happens |
|---|---|---|---|
| 0.1 | 0.8 | 1, 0.8, 0.64, 0.512 | shrinks slowly: too small |
| 0.45 | 0.1 | 1, 0.1, 0.01, 0.001 | fast |
| 0.9 | −0.8 | 1, −0.8, 0.64, −0.512 | jumps across the minimum every step, but shrinks |
| 1.05 | −1.1 | 1, −1.1, 1.21, −1.331 | grows: diverges |

🖼 **Picture.** The four cases on the parabola.

In [ ]:
fig_lr_effect()

**Proposition.** Gradient descent on $f(w) = \frac{\lambda}{2} w^2$ ($\lambda > 0$, the curvature) converges to the minimum from every starting point if and only if $0 < \eta < \frac{2}{\lambda}$. It's fastest at $\eta = \frac{1}{\lambda}$, which reaches the minimum in one step.

**Proof.** The gradient is $\lambda w$, so $w_{t+1} = w_t - \eta \lambda w_t = (1 - \eta\lambda)\, w_t$, hence $w_t = (1 - \eta\lambda)^t\, w_0$. This goes to 0 for every $w_0$ exactly when $\lvert 1 - \eta\lambda \rvert < 1$, i.e. $0 < \eta\lambda < 2$. At $\eta = 1/\lambda$ the factor is 0. $\blacksquare$

**Remarks.** (1) For $f(w) = w^2$, $\lambda = 2$, so $\eta < 1$: the table's 1.05 diverges. (2) **In many dimensions**, every direction has its own curvature (the eigenvalues $\lambda_{\min}, \dots, \lambda_{\max}$ of the Hessian). The step must satisfy $\eta < 2 / \lambda_{\max}$ or the steep direction blows up, but then the flat direction shrinks only by $(1 - \eta\lambda_{\min})$ per step. For §E's valley, $\lambda_{\max} = 100$ forces $\eta < 0.02$, and the flat direction ($\lambda = 1$) then loses only 2% per step: hence 235 steps. The ratio $\lambda_{\max} / \lambda_{\min}$ (the condition number) sets how slow gradient descent is. (3) In practice: if the loss goes up or becomes `nan`, the learning rate is too large; if it barely moves, too small. Try values spaced by factors of 3 to 10.

**Exercise 2 · Gradient descent, every iterate.** `grad` is a function returning the gradient at `w`. Starting from `w0` (a number or an array), take `steps` steps $w \leftarrow w - \eta\, \nabla f(w)$ and return all iterates **including `w0`** as one array, **shape `(steps + 1,) + w0.shape`**. Don't change `w0`: start from a copy (`np.array(w0, dtype=float)` makes one).

**Example**: what goes in, and what should come out:
```python
gd_steps(lambda w: 2 * w, 1.0, 0.1, 3)                    # → [1.    0.8   0.64  0.512]
gd_steps(lambda w: 2 * w, np.array([1.0, -2.0]), 0.45, 2)
# → [[ 1.   -2.  ]
#    [ 0.1  -0.2 ]
#    [ 0.01 -0.02]]
```

In [ ]:
def gd_steps(grad, w0, lr, steps):
    pass  # your code

In [ ]:
check(2)

## 2. Minibatches: stochastic gradient descent (SGD)

**The question:** the loss is an average over $n$ training samples, $L(w) = \frac{1}{n}\sum_{i=1}^{n} \ell_i(w)$, so its gradient costs $n$ per-sample gradients. With $n$ = 60,000 images, one step = one pass over all the data. Can a few samples stand in for all of them?

**Proposition.** Let $B$ be a uniformly random subset of $b$ of the $n$ samples, and $g_B = \frac{1}{b}\sum_{i \in B} \nabla \ell_i(w)$. Then $\mathbb{E}[g_B] = \nabla L(w)$: the minibatch gradient is right **on average** (unbiased).

**Proof.** Write $g_B = \frac{1}{b}\sum_{i=1}^{n} \mathbb{1}[i \in B]\, \nabla \ell_i(w)$. Each sample is in $B$ with probability $\frac{b}{n}$, so by linearity of expectation

$$
\mathbb{E}[g_B] = \frac{1}{b}\sum_{i=1}^{n} P(i \in B)\, \nabla \ell_i(w) = \frac{1}{b} \cdot \frac{b}{n}\sum_{i=1}^{n} \nabla \ell_i(w) = \nabla L(w). \qquad \blacksquare
$$

**Remarks.** (1) Unbiased doesn't mean exact: each $g_B$ is noisy, and the noise shrinks like $1 / \sqrt{b}$. (2) A step now costs $b$ instead of $n$, so one pass over the data gives $\lceil n / b \rceil$ steps instead of 1. That's the trade: many cheap, noisy steps beat one exact expensive one. (3) The noise even helps a little: it can shake the weights out of poor flat regions. (4) "SGD" strictly means $b = 1$, but in practice it means minibatches, typically $b$ = 32 to 256.

**Algorithm card: minibatch SGD**

**Goal.** Minimize $L(w) = \frac{1}{n}\sum_i \ell_i(w)$ without computing the full gradient.

- For each epoch $e = 1, \dots, E$: shuffle the sample indices (a random permutation).
- Cut the permutation into consecutive batches of $b$ indices (the last batch may be smaller).
- For each batch $B$ (step $t$): compute $g_B$ on those samples only, then $w^{(t+1)} = w^{(t)} - \eta\, g_B$.

**Why it works:** every step is right on average (the proposition), and with a small enough (or decreasing, §6) learning rate the noise averages out. **Why shuffle every epoch:** so the batches differ each time, and no ordering in the data (e.g. all cats first) biases a run of steps. **Caveats:** the loss curve is jagged; the learning rate and $b$ interact (a larger batch is less noisy and tolerates a larger $\eta$). **Cost per epoch:** the same number of per-sample gradients as one full-batch step, but $\lceil n / b \rceil$ updates. **Example trace:** $n = 10$, $b = 4$:

In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(10)                   # 0..9 in a random order, each exactly once
batches = [perm[start:start + 4] for start in range(0, 10, 4)]    # start = 0, 4, 8
print(batches)    # → [array([4, 6, 2, 7]), array([3, 5, 9, 0]), array([8, 1])]

🖼 **Picture.** Linear regression on 512 samples, trained three ways for 5 epochs. The full-batch run takes 5 steps; $b = 32$ takes 80; $b = 1$ takes 2560 noisy steps (with a smaller $\eta$, since single samples are noisy). For the same amount of data processed, the minibatch runs get much further.

In [ ]:
fig_sgd_noise()

**Exercise 3 · Cut an epoch into minibatches.** Call `rng.permutation(n)` **once**, then cut it into consecutive pieces of length `b` (the last may be shorter). Return the list of index arrays. (Calling `rng` once, in this way, lets the check reproduce your batches.)

**Example**: what goes in, and what should come out:
```python
minibatches(10, 4, np.random.default_rng(0))    # → [array([4, 6, 2, 7]), array([3, 5, 9, 0]), array([8, 1])]
```

In [ ]:
def minibatches(n, b, rng):
    pass  # your code

In [ ]:
check(3)

## 3. Momentum: average the directions

**The question:** in the valley, why does gradient descent zigzag, and how do we stop it? Across the valley the gradient flips sign every step (the steps overshoot); along the valley it points the same way every step. **Averaging** recent gradients cancels the flipping part and keeps the consistent part.

$$
v_t = \beta\, v_{t-1} + g_t, \qquad w_t = w_{t-1} - \eta\, v_t, \qquad v_0 = 0 \qquad (\beta = 0.9 \text{ typically})
$$

**Proposition.** Unrolled, $v_t = \sum_{s=1}^{t} \beta^{t-s} g_s$. In particular, if every gradient equals $g$, then $v_t = \frac{1 - \beta^t}{1 - \beta}\, g \to \frac{g}{1 - \beta}$.

**Proof** (induction on $t$). $v_1 = \beta \cdot 0 + g_1 = g_1$. If $v_{t-1} = \sum_{s=1}^{t-1} \beta^{t-1-s} g_s$, then

$$
v_t = \beta \sum_{s=1}^{t-1} \beta^{t-1-s} g_s + g_t = \sum_{s=1}^{t-1} \beta^{t-s} g_s + \beta^0 g_t = \sum_{s=1}^{t} \beta^{t-s} g_s.
$$

With all $g_s = g$ this is $g \sum_{k=0}^{t-1} \beta^k = g\, \frac{1 - \beta^t}{1 - \beta}$ (a geometric series). $\blacksquare$

**Remarks.** (1) In a consistent direction the step grows to $\frac{1}{1 - \beta}$ times the plain step: 10 times for $\beta = 0.9$. (2) A gradient that flips sign, $g, -g, g, \dots$, keeps $v$ small (about $\frac{g}{1 + \beta}$): the zigzag is damped. (3) $\beta^k$ is the weight of a gradient $k$ steps old, so the memory is about $\frac{1}{1 - \beta}$ steps. (4) Some books write $v \leftarrow \beta v - \eta g$, $w \leftarrow w + v$; with a fixed $\eta$ it's the same method. PyTorch uses the form above. (`nesterov=True` is a variant that looks one step ahead.)

🖼 **Picture.** Left: a gradient that zigzags between about $+2.6$ and $-1.4$ (a flip of $\pm 2$ around a steady $0.6$). The momentum average (scaled by $1 - \beta$) sits near $0.6$: the zigzag cancels, the steady part survives. Right: how much each past gradient counts.

In [ ]:
fig_ewma()

**Drill (on paper).** Momentum on $f(w) = w^2$ from $w_0 = 1$, with $\eta = 0.1$ and $\beta = 0.9$. Compute $g_t$, $v_t$, $w_t$ for $t = 1, 2, 3$, and compare with plain gradient descent (0.8, 0.64, 0.512).

<details><summary>Worked answer</summary>

$t = 1$: $g = 2$, $v = 2$, $w = 1 - 0.2 = 0.8$. $\ t = 2$: $g = 1.6$, $v = 0.9 \cdot 2 + 1.6 = 3.4$, $w = 0.8 - 0.34 = 0.46$. $\ t = 3$: $g = 0.92$, $v = 0.9 \cdot 3.4 + 0.92 = 3.98$, $w = 0.46 - 0.398 = 0.062$. Momentum is much faster here (0.062 vs 0.512); the velocity keeps it moving, and on later steps it will overshoot past 0 before settling.

</details>

**Exercise 4 · One momentum step.** $v \leftarrow \beta v + g$, then $w \leftarrow w - \eta v$. Return `(w_new, v_new)` as **new** arrays (don't change the inputs). `w`, `v`, `g` have the same shape.

**Example**: what goes in, and what should come out:
```python
momentum_step(np.array([0.8]), np.array([2.0]), np.array([1.6]), 0.1, 0.9)    # → (array([0.46]), array([3.4]))   the drill's t = 2
```

In [ ]:
def momentum_step(w, v, g, lr, beta):
    pass  # your code

In [ ]:
check(4)

## 4. RMSProp: a step size for every weight

**The question:** in the valley, the steep direction has large gradients and the flat one small ones. One shared $\eta$ is too large for one or too small for the other. What if each weight's step were divided by the typical size of **its own** recent gradients?

$$
s_t = \rho\, s_{t-1} + (1 - \rho)\, g_t^2, \qquad w_t = w_{t-1} - \eta\, \frac{g_t}{\sqrt{s_t} + \varepsilon} \qquad \text{(entry by entry; } \rho = 0.99,\ \varepsilon = 10^{-8})
$$

**What it does.** $s_t$ is an EWMA of the **squared** gradients, so $\sqrt{s_t}$ is a typical gradient size (a root-mean-square, hence the name). Dividing by it makes every weight move about $\eta$ per step, whatever the scale of its gradients: in the valley, the steep direction gets small steps and the flat direction large ones. $\varepsilon$ only prevents division by zero.

**A failure at the start.** $s_0 = 0$, so $s_1 = (1 - \rho)\, g_1^2$ and $\sqrt{s_1} = \sqrt{0.01}\, \lvert g_1 \rvert = 0.1 \lvert g_1 \rvert$. The first step is $\eta\, \frac{g_1}{0.1 \lvert g_1 \rvert} = \pm 10\eta$: ten times too large. That's the big jump at the start of RMSProp's path in §E. Adam fixes it (§5).

**Exercise 5 · One RMSProp step.** $s \leftarrow \rho s + (1 - \rho) g^2$, then $w \leftarrow w - \eta\, \frac{g}{\sqrt{s} + \varepsilon}$, entry by entry. Return `(w_new, s_new)` as new arrays.

**Example**: what goes in, and what should come out:
```python
rmsprop_step(np.array([1.0]), np.zeros(1), np.array([2.0]), 0.01)    # → (array([0.9]), array([0.04]))   a first step of 10 × 0.01
```

In [ ]:
def rmsprop_step(w, s, g, lr, rho=0.99, eps=1e-8):
    pass  # your code

In [ ]:
check(5)

## 5. Adam: momentum + RMSProp + bias correction

**Algorithm card: Adam**

**Goal.** Steps with momentum's direction (an average gradient) and RMSProp's per-weight scale, correct from the very first step.

**Steps.** Start with $m_0 = s_0 = 0$. For $t = 1, 2, \dots$:

$$
\begin{aligned} g_t &= \nabla L(w_{t-1}) \\ m_t &= \beta_1 m_{t-1} + (1 - \beta_1)\, g_t && \text{(EWMA of gradients: direction)} \\ s_t &= \beta_2 s_{t-1} + (1 - \beta_2)\, g_t^2 && \text{(EWMA of squared gradients: scale)} \\ \hat m_t &= \frac{m_t}{1 - \beta_1^t}, \qquad \hat s_t = \frac{s_t}{1 - \beta_2^t} && \text{(bias correction)} \\ w_t &= w_{t-1} - \eta\, \frac{\hat m_t}{\sqrt{\hat s_t} + \varepsilon} \end{aligned}
$$

Defaults: $\eta = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$.

**Why it works.** $\hat m_t$ estimates the average gradient (momentum's damping of zigzags); $\sqrt{\hat s_t}$ estimates its typical size. Their ratio is about $\pm 1$ in a consistent direction and small where the gradient keeps flipping, so each weight moves at most about $\eta$ per step.

**Why the correction: Proposition.** If the gradients $g_s$ all have the same mean $\mathbb{E}[g]$, then $\mathbb{E}[m_t] = (1 - \beta_1^t)\, \mathbb{E}[g]$; so $\hat m_t = m_t / (1 - \beta_1^t)$ has mean $\mathbb{E}[g]$.

**Proof.** Unrolling as in §3 (now with the factor $1 - \beta_1$): $m_t = (1 - \beta_1)\sum_{s=1}^{t} \beta_1^{t-s} g_s$. Taking expectations,

$$
\mathbb{E}[m_t] = (1 - \beta_1)\, \mathbb{E}[g] \sum_{k=0}^{t-1} \beta_1^{k} = (1 - \beta_1)\, \mathbb{E}[g]\, \frac{1 - \beta_1^t}{1 - \beta_1} = (1 - \beta_1^t)\, \mathbb{E}[g]. \qquad \blacksquare
$$

**Remarks.** (1) The same argument holds for $s_t$ with $\beta_2$. (2) Without the correction, at $t = 1$: $m_1 = 0.1 g$ and $\sqrt{s_1} = 0.0316 \lvert g \rvert$, a ratio of about $3.16$; with it, $\hat m_1 / \sqrt{\hat s_1} = g / \lvert g \rvert = \pm 1$, so the **first step has size exactly $\eta$**. (3) The step size barely depends on the scale of the loss, which is why Adam works with the same $\eta = 0.001$ on very different problems. (4) **Caveats:** two extra arrays per parameter ($m$, $s$), so three times the parameter memory; on some vision tasks a well-tuned SGD with momentum generalizes slightly better. AdamW (Week 6) adds weight decay the right way. **Cost:** a few operations per weight per step.

**Example trace**, $f(w) = w^2$ from $w_0 = 1$ with $\eta = 0.1$:

| $t$ | $g_t$ | $m_t$ | $s_t$ | $\hat m_t$ | $\hat s_t$ | $w_t = w_{t-1} - 0.1\, \hat m_t / \sqrt{\hat s_t}$ |
|---|---|---|---|---|---|---|
| 1 | 2 | 0.2 | 0.004 | 0.2 / 0.1 = 2 | 0.004 / 0.001 = 4 | 1 − 0.1 · 2 / 2 = **0.9** |
| 2 | 1.8 | 0.9 · 0.2 + 0.1 · 1.8 = 0.36 | 0.999 · 0.004 + 0.001 · 3.24 = 0.007236 | 0.36 / 0.19 = 1.8947 | 0.007236 / 0.001999 = 3.6198 | 0.9 − 0.1 · 1.8947 / 1.9026 = **0.80041** |

**Exercise 6 · One Adam step.** Apply the card once, at step number `t` (counting from 1). Return `(w_new, m_new, s_new)` as new arrays.

**Example**: what goes in, and what should come out:
```python
w, m, s = adam_step(np.array([1.0]), np.zeros(1), np.zeros(1), np.array([2.0]), 1, lr=0.1)
w, m, s    # → (array([0.9]), array([0.2]), array([0.004]))
adam_step(w, m, s, 2 * w, 2, lr=0.1)[0]    # → [0.80041]
```

In [ ]:
def adam_step(w, m, s, g, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    pass  # your code

In [ ]:
check(6)

## 6. Learning-rate schedules

**The question:** should $\eta$ stay fixed? Early in training, large steps make fast progress; late, they keep the weights bouncing around the minimum (with SGD, the noise floor is proportional to $\eta$). So most training runs **lower** the learning rate over time.

🖼 **Picture.** Four common schedules over 100 epochs, starting from $\eta = 0.1$. Warm-up (orange) starts small for the first epochs, then decays; it's standard for Transformers and large batch sizes, where big early steps from random weights can wreck training.

In [ ]:
fig_schedules()

In [ ]:
w = torch.tensor([1.0], requires_grad=True)
opt = torch.optim.SGD([w], lr=0.1)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.1)   # lr x 0.1 every 30 epochs
lrs = []
for epoch in range(90):
    # ... one epoch of training: opt.zero_grad(); loss.backward(); opt.step() for every batch ...
    opt.step()                       # (here just once, so the scheduler has a step to follow)
    sched.step()                     # once per EPOCH, after the optimizer's steps
    lrs.append(opt.param_groups[0]["lr"])
print([round(lrs[i], 4) for i in (0, 29, 30, 59, 60)])    # → [0.1, 0.01, 0.01, 0.001, 0.001]

Read the output: `lrs[i]` is the rate **after** epoch $i$'s `sched.step()`. After 30 steps of the scheduler (index 29) the rate has dropped to 0.01, after 60 to 0.001.

## 7. `torch.optim`, number for number, and your own Adam

The hand calculations from §3 and §5, in PyTorch. `opt.step()` reads each parameter's `.grad` and changes the parameter **in place**:

In [ ]:
def three_steps(opt_class, steps, **settings):
    w = torch.tensor(1.0, requires_grad=True)       # f(w) = w², from w = 1
    opt = opt_class([w], **settings)
    out = []
    for _ in range(steps):
        opt.zero_grad()
        (w ** 2).backward()                         # w.grad = 2w
        opt.step()
        out.append(round(w.item(), 5))
    return out

print(three_steps(torch.optim.SGD, 3, lr=0.1, momentum=0.9))   # → [0.8, 0.46, 0.062]   the drill in §3
print(three_steps(torch.optim.Adam, 2, lr=0.1))                # → [0.9, 0.80041]       the table in §5
print(three_steps(torch.optim.RMSprop, 2, lr=0.01))            # → [0.9, 0.83292]       a first step of 10 × 0.01

**Your own Adam, as PyTorch organizes it.** An optimizer object is given the model's parameter arrays once, keeps its own state ($m$, $s$, $t$), and `step(grads)` changes the parameters **in place**. This is the one place where changing the caller's arrays is the whole point (contrast with `00` A2): the model holds those same arrays, so an in-place update is how it sees the new weights. Writing `p = p - ...` would only rebind a local name, and the model would never change.

| `MyAdam` attribute | Holds |
|---|---|
| `self.params` | the list of the model's arrays (the same objects, not copies) |
| `self.m`, `self.s` | one zero array per parameter, same shapes |
| `self.t` | the number of steps taken so far (starts at 0) |
| `step(grads)` | `self.t += 1`, then the Adam update for each `(p, g)` pair, writing into `p` with `p -= ...` |

**Exercise 7 · Your Adam class.** `MyAdam(params, lr=0.001, betas=(0.9, 0.999), eps=1e-8)` and `step(grads)` as in the table: `grads` is a list of arrays, one per parameter, in the same order. Update each parameter **in place** (`p -= ...`); the check verifies that the model's arrays themselves changed.

**Example**: what goes in, and what should come out:
```python
w = np.array([1.0])                 # the 'model' holds this array
opt = MyAdam([w], lr=0.1)
opt.step([2 * w]); opt.step([2 * w])
w                                   # → [0.80041]   the same array object, changed in place
```

In [ ]:
class MyAdam:
    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), eps=1e-8):
        pass  # your code

    def step(self, grads):
        pass  # your code

In [ ]:
check(7)

After exercise 7 passes, race your Adam against PyTorch's on the same two arrays:

In [ ]:
W = np.array([[1.0, -2.0], [0.5, 3.0]]); b = np.array([0.1, -0.1])          # a "model" with two arrays
Wt = torch.tensor(W, requires_grad=True); bt = torch.tensor(b, requires_grad=True)
mine, theirs = MyAdam([W, b], lr=0.05), torch.optim.Adam([Wt, bt], lr=0.05)
for step in range(5):                                # minimize sum(W**2) + sum(b**2) both ways
    mine.step([2 * W, 2 * b])
    theirs.zero_grad(); ((Wt ** 2).sum() + (bt ** 2).sum()).backward(); theirs.step()
print(np.allclose(W, Wt.detach().numpy()), np.allclose(b, bt.detach().numpy()))   # → True True

**Recipe (choosing an optimizer).** Start with Adam at $\eta = 10^{-3}$. If the loss is erratic or `nan`, lower $\eta$; if it falls very slowly, raise it, trying factors of 3. For long runs add a schedule (cosine, or step decay). SGD with momentum 0.9 and a schedule is the main alternative, common for CNNs.

In [ ]:
for n in range(1, 8):
    check(n)

---
Solutions: `solutions/05_solutions.py`. Next: `06_pytorch_basics.ipynb`, where all of this becomes ordinary PyTorch.